https://youtu.be/hQfg2nLH1gA?si=7keRLoCISirF0KJ4

In [0]:
from pyspark.sql import functions as F
from datetime import date
from pyspark.sql.window import Window

data = [
    ("success", date(2025, 8, 1)),
    ("success", date(2025, 8, 2)),
    ("fail", date(2025, 8, 3)),
    ("fail", date(2025, 8, 4)),
    ("success", date(2025, 8, 13))
]

df = spark.createDataFrame(data, ["status", "date"])
df.display()

status,date
success,2025-08-01
success,2025-08-02
fail,2025-08-03
fail,2025-08-04
success,2025-08-13


In [0]:
df1 = (
    df.withColumn("rank1",F.row_number().over(Window.partitionBy("status").orderBy(F.col("date"))))
    .withColumn("rank2", F.row_number().over(Window.orderBy(F.col("date"))))
    .withColumn("diff",F.expr("rank1 - rank2"))
    .groupBy("status","diff")
    .agg(
        F.min("date").alias("start_date"),
        F.max("date").alias("end_date")        
    )
    .drop("diff")
)

df1.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


status,start_date,end_date
success,2025-08-01,2025-08-02
fail,2025-08-03,2025-08-04
success,2025-08-13,2025-08-13
